> 在线讲义：[第 1 章 · 智能体与环境 agent & environment](https://wikieden.github.io/rl-from-scratch/lessons/01-agent-environment.html) · 本书《从零学强化学习》

# 第 1 章 · 智能体与环境 agent & environment

本章回答：强化学习到底在"学"什么？谁在学、跟谁打交道？它和"有标准答案"的学习有什么不同？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/01-agent-environment.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 环境 | `class DogEnv` 的 `step(action)` 返回 `(奖励, 新状态)` | — |
| 轨迹 | `rollout(env, ["坐下", "乱跑"])` | `S0=主人举手, A0=坐下, R1=1, S1=主人放下手, A1=乱跑, R2=0, S2=主人举手` |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**类 `DogEnv`**（来自 `ch01_agent_environment.py`）：讲义的训练小狗：主人举手时坐下得 1 块零食，其余 0。主人举手 / 放手轮流出现。

In [ ]:
class DogEnv:
    """讲义的训练小狗：主人举手时坐下得 1 块零食，其余 0。主人举手 / 放手轮流出现。"""

    def __init__(self):
        self.state = "主人举手"

    def step(self, action):
        reward = 1 if (self.state == "主人举手" and action == "坐下") else 0
        self.state = "主人放下手" if self.state == "主人举手" else "主人举手"
        return reward, self.state

**函数 `rollout`**（来自 `ch01_agent_environment.py`）：按给定动作走几步，返回轨迹列表 [("S0", ...), ("A0", ...), ("R1", ...), ...]。

In [ ]:
def rollout(env, actions):
    """按给定动作走几步，返回轨迹列表 [("S0", ...), ("A0", ...), ("R1", ...), ...]。"""
    traj = [("S0", env.state)]
    for t, a in enumerate(actions):
        traj.append((f"A{t}", a))
        r, s = env.step(a)
        traj.append((f"R{t + 1}", r))  # 奖励的下标比动作大 1
        traj.append((f"S{t + 1}", s))
    return traj

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
traj = rollout(DogEnv(), ["坐下", "乱跑"])

In [ ]:
print("轨迹：", ", ".join(f"{k}={v}" for k, v in traj))

## 3. 代码里才看得到的

`step()` 这个函数名就是后面所有 RL 代码库（Gymnasium、CS 285 作业）的标准接口：智能体把动作交给 `env.step(a)`，环境回给它奖励和新状态。第 2 个动作"乱跑"之后 `R2 = 0`，下标正好比动作大 1。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换一串动作：`rollout(DogEnv(), ["乱跑", "坐下", "坐下"])`。第一次坐下时主人已经放下手，`R2 = 0`；第二次坐下赶上举手，`R3 = 1`。同一个动作拿不拿奖励，要看当时的状态。
2. 在 `DogEnv.step` 里把奖励改成 `2`，再跑一遍：轨迹的形状不变，只是 R 的数字变了。环境定规则，智能体只能选动作。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 看状态，做动作，拿奖励，再看新状态；
> 奖励下标大一号；
> 监督学习给答案，强化学习只给分；改不了的都是环境。